# Phase 4: Safe LLM Feature Planning
This notebook demonstrates structured JSON feature planning with Ollama and deterministic materialization without executing model-generated code.

In [ ]:
from pprint import pprint

from retail_forecasting.feature_spec import extract_raw_specs_from_payload, validate_feature_plan_specs
from retail_forecasting.features_llm import (
    build_llm_features_pipeline,
    load_llm_source_data,
    load_llm_features_summary,
    load_manual_feature_namespace,
    materialize_llm_features,
    render_feature_planner_prompts,
)
from retail_forecasting.ollama_client import OllamaClient

In [ ]:
source_path, source_df = load_llm_source_data()
manual_namespace = load_manual_feature_namespace()

print("Source path:", source_path)
print("Shape:", source_df.shape)
print("Available columns:")
print(sorted(source_df.columns.tolist()))

In [ ]:
system_prompt, user_prompt = render_feature_planner_prompts(
    available_columns=source_df.columns.tolist(),
    manual_feature_names=sorted(manual_namespace),
)

print("System prompt chars:", len(system_prompt))
print("User prompt chars:", len(user_prompt))
print("User prompt preview:")
print(user_prompt[:600])

In [ ]:
client = OllamaClient()
planner_response = client.plan_feature_specs(system_prompt=system_prompt, user_prompt=user_prompt)

print("Ollama reachable:", planner_response.reachable)
print("Planner model:", planner_response.model)
print("Planner error:", planner_response.error)
print("Raw response preview:")
print((planner_response.raw_response_text or "")[:800])

In [ ]:
raw_payload = planner_response.parsed_json or {"specs": []}
raw_specs = extract_raw_specs_from_payload(raw_payload)

accepted_specs, rejected_specs = validate_feature_plan_specs(
    raw_specs=raw_specs,
    available_columns=source_df.columns.tolist(),
    existing_feature_names=source_df.columns.tolist(),
    blocked_feature_names=sorted(manual_namespace),
)

print("Raw specs:", len(raw_specs))
print("Accepted specs:", len(accepted_specs))
print("Rejected specs:", len(rejected_specs))
pprint(rejected_specs[:5])

In [ ]:
if accepted_specs:
    llm_feature_frame, created_features, materialization_rejections = materialize_llm_features(
        source_df=source_df,
        specs=accepted_specs,
    )
else:
    llm_feature_frame = source_df.copy()
    created_features = []
    materialization_rejections = []

print("Created features:", len(created_features))
print(created_features)
print("Materialization rejections:", len(materialization_rejections))
pprint(materialization_rejections[:5])
llm_feature_frame.head()

In [ ]:
artifacts = build_llm_features_pipeline()
print("Written artifacts:")
for name, path in artifacts.items():
    print(name, "->", path)

summary = load_llm_features_summary()
print("Summary counts:")
print("raw_spec_count:", summary.get("raw_spec_count"))
print("accepted_spec_count:", summary.get("accepted_spec_count"))
print("rejected_spec_count:", summary.get("rejected_spec_count"))
print("output_feature_count:", summary.get("output_feature_count"))